## 1. Setup and Environment

In [2]:
import sys
sys.path.append(r"D:\Quant\Multi_factor_project")
from config import *

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("OK Environment loaded successfully")

✓ 环境加载完成


## 2. Load and Merge Core Data Inputs

Load the base monthly panel, daily prices, adjustment factors (to build `adj_close`), financial statements, valuation data, and industry classification.

In [3]:
# Base panel
base_panel = pd.read_parquet(f"{DATA_CLEAN}/base_panel.parquet")
base_panel['month'] = base_panel['month'].dt.to_timestamp()

# Daily price panel
daily_panel = pd.read_parquet(f"{DATA_CLEAN}/daily_panel.parquet")
daily_panel['trade_date'] = pd.to_datetime(daily_panel['trade_date'])

# Adjustment factor -> construct post-rights-adjusted close price (adj_close is used
# consistently across the whole project for return and momentum calculations)
adj_dir = f"{DATA_RAW}/adj_factor"
adj = pd.concat(
    [pd.read_parquet(os.path.join(adj_dir, f)) for f in os.listdir(adj_dir) if f.endswith('.parquet')],
    ignore_index=True
)
adj['trade_date'] = pd.to_datetime(adj['trade_date'])
daily_panel = daily_panel.merge(adj[['ts_code','trade_date','adj_factor']],
                                on=['ts_code','trade_date'], how='left')
daily_panel['adj_close'] = daily_panel['close'] * daily_panel['adj_factor']
print(f"adj_factor missing rate: {daily_panel['adj_factor'].isna().mean()*100:.2f}%")  # should be near 0

# Financials and valuation
daily_basic = pd.read_parquet(f"{DATA_RAW}/financial/daily_basic.parquet")
daily_basic['trade_date'] = pd.to_datetime(daily_basic['trade_date'])
income  = pd.read_parquet(f"{DATA_RAW}/financial/income.parquet")
balance = pd.read_parquet(f"{DATA_RAW}/financial/balancesheet.parquet")

# Industry classification
industry_map = pd.read_parquet(f"{DATA_RAW}/industry/sw_industry_map.parquet")
industry_map['in_date']  = pd.to_datetime(industry_map['in_date'], errors='coerce')
industry_map['out_date'] = pd.to_datetime(industry_map['out_date'], errors='coerce')

print(f"OK base_panel: {base_panel.shape}")     # should be 30k+ rows

adj_factor 缺失率：0.00%
✓ base_panel: (35255, 4)


## 3. Point-in-Time Industry Classification

Assign each (stock, month) to a Shenwan Level-1 industry using `merge_asof` on the industry effective date, so that only classification information already in effect at that month is used. Early months with no classification record are backfilled from each stock's earliest known industry (a documented approximation).

In [4]:
def get_monthly_industry(industry_map, base_panel):
    \"\"\"merge_asof: for each (stock, month), match the most recent industry record
    whose in_date is <= the current month.\"\"\"
    keys = (base_panel[['ts_code','month']].drop_duplicates()
              .sort_values('month').reset_index(drop=True))
    ind = (industry_map[['con_code','in_date','out_date','industry_name','industry_code']]
             .dropna(subset=['in_date']).sort_values('in_date'))

    merged = pd.merge_asof(keys, ind,
                           left_on='month', right_on='in_date',
                           by=None, left_by='ts_code', right_by='con_code',
                           direction='backward')

    # Industry membership has already expired (out_date is before the current month) -> set to NaN
    expired = merged['out_date'].notna() & (merged['out_date'] <= merged['month'])
    merged.loc[expired, ['industry_name','industry_code']] = np.nan

    # Early months have no record (SW2021 classification was published later) -> backfill
    # using the stock's earliest available industry record.
    # This effectively assumes "industry membership is constant over history" - a common
    # approximation, and one to stay aware of.
    first_ind = ind.groupby('con_code')['industry_name'].first()
    merged['industry_name'] = merged['industry_name'].fillna(merged['ts_code'].map(first_ind))

    print(f"Industry classification complete: {len(merged)} records, coverage: {merged['industry_name'].notna().mean()*100:.1f}%")
    return merged[['ts_code','month','industry_name']]

monthly_industry = get_monthly_industry(industry_map, base_panel)

行业分类完成：35255 条，覆盖率：100.0%


## 4. Momentum and Reversal Factors

Construct 12-month momentum (skip-month, following Jegadeesh & Titman 1993) and 1-month reversal from adjusted monthly close prices.

In [5]:
def calc_momentum_factor(daily_panel, base_panel):
    \"\"\"mom_12m: cumulative return from month t-14 to t-2 (12-month momentum,
                  skipping the most recent 1 month)
       mom_1m : prior month's return (short-term reversal)\"\"\"
    daily = daily_panel[['ts_code','trade_date','adj_close']].copy()
    daily['month'] = daily['trade_date'].dt.to_period('M').dt.to_timestamp()

    monthly_close = (daily.sort_values('trade_date')
                          .groupby(['ts_code','month'])['adj_close'].last()
                          .reset_index().sort_values(['ts_code','month']))

    g = monthly_close.groupby('ts_code')['adj_close']
    monthly_close['mom_12m'] = g.shift(2) / g.shift(14) - 1
    monthly_close['mom_1m']  = g.shift(1) / g.shift(2) - 1

    result = base_panel[['ts_code','month']].merge(
        monthly_close[['ts_code','month','mom_12m','mom_1m']],
        on=['ts_code','month'], how='left')

    print(f"mom_12m missing rate: {result['mom_12m'].isna().mean()*100:.1f}%")  # higher than before is expected: recently-listed stocks have <14 months of history
    print(f"mom_1m  missing rate: {result['mom_1m'].isna().mean()*100:.1f}%")
    return result

momentum_factors = calc_momentum_factor(daily_panel, base_panel)

mom_12m 缺失率：12.4%
mom_1m  缺失率：0.9%


## 5. Quality Factors: ROE_TTM and Net Margin_TTM

Construct trailing-twelve-month profitability measures from cumulative (year-to-date) Chinese financial statements, matched point-in-time via `merge_asof` on announcement date to avoid look-ahead bias.

In [6]:
def calc_quality_factor(income, balance, base_panel):
    \"\"\"ROE_TTM and net profit margin_TTM.
       A-share financial statements report cumulative (year-to-date) figures:
       TTM = current YTD + prior full year - prior-year same-period YTD\"\"\"

    # ---- 1. Income statement -> TTM ----
    inc = income[['ts_code','end_date','ann_date','n_income_attr_p','revenue']].copy()
    inc['ann_date'] = pd.to_datetime(inc['ann_date'], errors='coerce')
    inc['end_date'] = pd.to_datetime(inc['end_date'], errors='coerce')
    inc = inc.dropna(subset=['ann_date','end_date'])
    inc['year'] = inc['end_date'].dt.year
    inc['md']   = inc['end_date'].dt.strftime('%m-%d')

    # Prior full year (year-end cumulative value)
    fy = inc[inc['md']=='12-31'][['ts_code','year','n_income_attr_p','revenue']].copy()
    fy['year'] += 1
    fy.columns = ['ts_code','year','fy_ni_ly','fy_rev_ly']
    inc = inc.merge(fy, on=['ts_code','year'], how='left')

    # Prior-year same-period cumulative value
    ly = inc[['ts_code','year','md','n_income_attr_p','revenue']].copy()
    ly['year'] += 1
    ly.columns = ['ts_code','year','md','ytd_ni_ly','ytd_rev_ly']
    inc = inc.merge(ly, on=['ts_code','year','md'], how='left')

    inc['ni_ttm']  = np.where(inc['md']=='12-31', inc['n_income_attr_p'],
                              inc['n_income_attr_p'] + inc['fy_ni_ly'] - inc['ytd_ni_ly'])
    inc['rev_ttm'] = np.where(inc['md']=='12-31', inc['revenue'],
                              inc['revenue'] + inc['fy_rev_ly'] - inc['ytd_rev_ly'])
    fin = (inc.dropna(subset=['ni_ttm'])
              .sort_values(['ann_date','end_date'])   # when multiple filings share an announcement date, the one with the newer reporting period sorts last
              [['ts_code','ann_date','ni_ttm','rev_ttm']])

    # ---- 2. Net assets (point-in-time value, no TTM needed) ----
    bal = balance[['ts_code','end_date','ann_date','total_hldr_eqy_exc_min_int']].copy()
    bal['ann_date'] = pd.to_datetime(bal['ann_date'], errors='coerce')
    bal['end_date'] = pd.to_datetime(bal['end_date'], errors='coerce')
    bal = (bal.dropna(subset=['ann_date'])
              .sort_values(['ann_date','end_date'])
              [['ts_code','ann_date','total_hldr_eqy_exc_min_int']])

    # ---- 3. merge_asof: only use filings already announced before the 1st of the month (no look-ahead) ----
    keys = (base_panel[['ts_code','month']].drop_duplicates()
              .sort_values('month').reset_index(drop=True))
    m_inc = pd.merge_asof(keys, fin, left_on='month', right_on='ann_date',
                          by='ts_code', direction='backward')
    m_bal = pd.merge_asof(keys, bal, left_on='month', right_on='ann_date',
                          by='ts_code', direction='backward')

    result = keys.copy()
    result['ni_ttm']  = m_inc['ni_ttm'].values
    result['rev_ttm'] = m_inc['rev_ttm'].values
    result['equity']  = m_bal['total_hldr_eqy_exc_min_int'].values

    # ROE is meaningless when net assets are negative -> NaN (more rigorous than the previous abs() approach)
    result['roe_ttm']    = np.where(result['equity'] > 0, result['ni_ttm'] / result['equity'], np.nan)
    result['net_margin'] = np.where(result['rev_ttm'] > 0, result['ni_ttm'] / result['rev_ttm'], np.nan)

    print(f"roe_ttm missing rate: {result['roe_ttm'].isna().mean()*100:.1f}%")
    print(f"net_margin missing rate: {result['net_margin'].isna().mean()*100:.1f}%")
    return result[['ts_code','month','roe_ttm','net_margin']]

quality_factors = calc_quality_factor(income, balance, base_panel)

roe_ttm 缺失率：2.0%
net_margin 缺失率：2.0%


## 6. Value Factors: PE_TTM and PB

Extract monthly PE_TTM and PB from daily valuation data, excluding economically meaningless negative values (loss-making firms, negative net assets).

In [7]:
def calc_value_factor(daily_basic, base_panel):
    db = daily_basic.copy()
    db['month'] = db['trade_date'].dt.to_period('M').dt.to_timestamp()

    monthly_val = (db.sort_values('trade_date')
                     .groupby(['ts_code','month']).last()
                     .reset_index()[['ts_code','month','pe_ttm','pb']])

    result = base_panel[['ts_code','month']].merge(monthly_val, on=['ts_code','month'], how='left')
    result.loc[result['pe_ttm'] <= 0, 'pe_ttm'] = np.nan  # loss-making stocks
    result.loc[result['pb'] <= 0, 'pb'] = np.nan          # negative net assets, also meaningless

    print(f"PE missing rate: {result['pe_ttm'].isna().mean()*100:.1f}%")
    print(f"PB missing rate: {result['pb'].isna().mean()*100:.1f}%")
    return result

value_factors = calc_value_factor(daily_basic, base_panel)

PE 缺失率：4.2%
PB 缺失率：0.0%


## 7. Merge into a Single Factor Panel

Combine momentum, quality, value, and industry classification into one monthly factor panel.

In [8]:
factor_panel = base_panel.copy()
factor_panel = factor_panel.merge(momentum_factors[['ts_code','month','mom_12m','mom_1m']], on=['ts_code','month'], how='left')
factor_panel = factor_panel.merge(quality_factors[['ts_code','month','roe_ttm','net_margin']], on=['ts_code','month'], how='left')
factor_panel = factor_panel.merge(value_factors[['ts_code','month','pe_ttm','pb']], on=['ts_code','month'], how='left')
factor_panel = factor_panel.merge(monthly_industry, on=['ts_code','month'], how='left')

print(f"Factor panel: {factor_panel.shape}")
print(list(factor_panel.columns))

因子面板：(35255, 11)
['ts_code', 'month', 'adj_close', 'ret_1m', 'mom_12m', 'mom_1m', 'roe_ttm', 'net_margin', 'pe_ttm', 'pb', 'industry_name']


## 8. Factor Distribution and Missing-Rate Check

Sanity-check the raw (pre-standardization) distribution and missing rate of each factor.

In [9]:
factors = ['mom_12m','mom_1m','roe_ttm','net_margin','pe_ttm','pb']
print("Missing rate and distribution for each factor:")
print(factor_panel[factors].describe().round(3))
print()
for f in factors:
    print(f"  {f}: missing {factor_panel[f].isna().mean()*100:.1f}%")

各因子缺失率与分布：
         mom_12m     mom_1m    roe_ttm  net_margin     pe_ttm         pb
count  30881.000  34954.000  34551.000   34541.000  33785.000  35254.000
mean       0.049      0.004      0.117       0.188     61.782      3.827
std        0.470      0.116      0.216       0.588   1020.284      4.836
min       -0.868     -0.678    -32.874      -4.309      1.393      0.287
25%       -0.220     -0.061      0.068       0.046     12.974      1.370
50%       -0.035     -0.005      0.109       0.120     23.256      2.407
75%        0.194      0.058      0.165       0.256     42.457      4.544
max       21.232      1.941      1.519      35.099  87341.971    301.726

  mom_12m: 缺失 12.4%
  mom_1m: 缺失 0.9%
  roe_ttm: 缺失 2.0%
  net_margin: 缺失 2.0%
  pe_ttm: 缺失 4.2%
  pb: 缺失 0.0%


## 9. Cross-Sectional Standardization (Industry-Neutral)

Apply MAD winsorization and Z-score normalization within each month x industry group, with a whole-market fallback for small industry groups, and sign-flip the value factors so higher standardized values consistently indicate a more attractive stock.

In [10]:
def standardize_factors(factor_panel, factors, min_group=5):
    \"\"\"Within each month x industry group: MAD-based winsorization + Z-score.
       If the industry group has fewer than min_group stocks, fall back to a
       whole-market Z-score for that month (avoids undefined/distorted values
       for small industries).\"\"\"
    df = factor_panel.copy()
    reverse = ['pe_ttm', 'pb']

    def mad_clip(x, n=3):
        med = x.median()
        mad = (x - med).abs().median()
        return x.clip(med - n*1.4826*mad, med + n*1.4826*mad) if mad > 0 else x

    for f in factors:
        # industry-month level
        v   = df.groupby(['month','industry_name'])[f].transform(mad_clip)
        cnt = v.groupby([df['month'], df['industry_name']]).transform('count')
        mu  = v.groupby([df['month'], df['industry_name']]).transform('mean')
        sd  = v.groupby([df['month'], df['industry_name']]).transform('std')
        z_ind = (v - mu) / sd

        # whole-market month level (fallback)
        vm  = df.groupby('month')[f].transform(mad_clip)
        z_m = (vm - vm.groupby(df['month']).transform('mean')) / vm.groupby(df['month']).transform('std')

        df[f'{f}_std'] = np.where((cnt >= min_group) & (sd > 0), z_ind, z_m)
        if f in reverse:
            df[f'{f}_std'] = -df[f'{f}_std']

    print("OK Standardization complete (small industries fall back to whole-market basis)")
    return df

factors_to_std = ['mom_12m','mom_1m','roe_ttm','net_margin','pe_ttm','pb']
factor_panel = standardize_factors(factor_panel, factors_to_std)

✓ 标准化完成（小行业已回退全市场口径）


## 10. Save Factor Panel

Persist the standardized factor panel for portfolio construction (Chapter 04).

In [11]:
factor_panel.to_parquet(f"{DATA_CLEAN}/factor_panel.parquet")
print(f"OK Save complete: {factor_panel.shape}")
for f in factors_to_std:
    print(f"  {f}_std missing rate: {factor_panel[f'{f}_std'].isna().mean()*100:.2f}%")

✓ 保存完成：(35255, 17)
  mom_12m_std 缺失率：12.41%
  mom_1m_std 缺失率：0.85%
  roe_ttm_std 缺失率：2.00%
  net_margin_std 缺失率：2.03%
  pe_ttm_std 缺失率：4.17%
  pb_std 缺失率：0.00%
